<img src="https://drive.usercontent.google.com/download?id=19NlKhM3FwlX-_9yCEnhZyLDPr4wicWmC&export=view" width="120" align="right"/>

# เอกสารประกอบการเรียนรู้
## หลักสูตร AI & Data Analytics (Machine Learning)
### บทที่ 5 — เฉลย Workshop + สรุปหลักสูตร

---

**โครงการ Coding Thailand 2026**
ระดับ: ผู้เริ่มต้น–ขั้นสูง | ระยะเวลา: 10 นาที | รูปแบบ: Interactive Notebook

---

**ผู้สอน**

**ผศ.ดร. สัญชัย เอียดปราบ**

อาจารย์ประจำหลักสูตรวิศวกรรมระบบสมองกลฝังตัวและอิเล็กทรอนิกส์สื่อสาร
คณะวิศวกรรมศาสตร์ มหาวิทยาลัยบูรพา

---

**วัตถุประสงค์ของบทนี้:**
1. ตรวจสอบคำตอบของ Workshop (บทที่ 4) ทุกระดับ
2. ทบทวนเนื้อหาทั้งหมดที่ได้เรียนรู้ในหลักสูตร
3. แนะนำแนวทางการศึกษาต่อในสาย AI

In [ ]:
# === เตรียมข้อมูลเหมือนบทที่ 4 ===
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
plt.rcParams['figure.figsize'] = (10, 5)

np.random.seed(2026)
n = 500
ph = np.random.uniform(4.0, 10.0, n)
turbidity = np.random.uniform(0, 20, n)
dissolved_oxygen = np.random.uniform(2, 12, n)
temperature = np.random.uniform(20, 38, n)
conductivity = np.random.uniform(100, 800, n)

score = (ph - 7)**2 * 5 + turbidity * 3 - dissolved_oxygen * 8 + abs(temperature - 28) * 2 + conductivity * 0.05
quality = np.where(score < np.median(score), "Safe", "Unsafe")

df = pd.DataFrame({
    "ph": np.where(np.random.random(n) < 0.05, np.nan, ph),
    "turbidity_ntu": np.where(np.random.random(n) < 0.03, np.nan, turbidity),
    "dissolved_oxygen": dissolved_oxygen,
    "temperature_c": temperature,
    "conductivity": conductivity,
    "quality": quality
})

df_clean = df.copy()
df_clean["ph"] = df_clean["ph"].fillna(df_clean["ph"].median())
df_clean["turbidity_ntu"] = df_clean["turbidity_ntu"].fillna(df_clean["turbidity_ntu"].median())

X = df_clean[["ph", "turbidity_ntu", "dissolved_oxygen", "temperature_c", "conductivity"]]
y = df_clean["quality"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("✅ ข้อมูลพร้อมแล้ว!")
print(f"   Train: {len(X_train)} ตัวอย่าง, Test: {len(X_test)} ตัวอย่าง")

---
## เฉลย Level 1

```python
# L1-1: Data Cleaning
df_clean["ph"] = df_clean["ph"].fillna(df_clean["ph"].median())
df_clean["turbidity_ntu"] = df_clean["turbidity_ntu"].fillna(df_clean["turbidity_ntu"].median())

# L1-2: เตรียมข้อมูล
y = df_clean["quality"]
test_size = 0.2

# L1-3: สร้าง + Train
model = DecisionTreeClassifier()
model.fit(X_train, y_train)

# L1-4: ทดสอบ
y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)
```

---
## เฉลย Level 2: เปรียบเทียบ 3 Models

In [ ]:
# L2-1: Train 3 Models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"  {name:<20} Accuracy: {acc:.1%}")

best = max(results, key=results.get)
print(f"\n🏆 Model ที่ดีที่สุด: {best} ({results[best]:.1%})")

In [ ]:
# L2-2: Bar Chart เปรียบเทียบ
plt.figure(figsize=(8, 5))
colors = ['#3498db', '#e74c3c', '#2ecc71']
bars = plt.bar(results.keys(), results.values(), color=colors)
for bar, score in zip(bars, results.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{score:.1%}', ha='center', fontsize=14, fontweight='bold')
plt.ylim(0, 1.1)
plt.ylabel('Accuracy')
plt.title('Model Comparison: Water Quality Classification')
plt.tight_layout()
plt.show()

In [ ]:
# L2-3: Feature Importance (Random Forest)
rf = models["Random Forest"]
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(8, 4))
importance.plot(kind='barh', color='#2ecc71')
plt.xlabel('Importance')
plt.title('Feature Importance (Random Forest)')
plt.tight_layout()
plt.show()

print(f"\n🔑 ค่าที่สำคัญที่สุด: {importance.idxmax()} ({importance.max():.3f})")

---
## ตัวอย่าง Level 3: Clustering น้ำ

In [ ]:
from sklearn.cluster import KMeans

# Clustering ข้อมูลน้ำ (ไม่ใช้ quality column)
X_cluster = df_clean[["ph", "turbidity_ntu", "dissolved_oxygen", "conductivity"]]

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
df_clean["cluster"] = kmeans.fit_predict(X_cluster)

# เปรียบเทียบ Cluster กับ Quality จริง
ct = pd.crosstab(df_clean["quality"], df_clean["cluster"], margins=True)
print("Cluster vs Actual Quality:")
print(ct)

plt.figure(figsize=(10, 5))
for c in [0, 1]:
    d = df_clean[df_clean["cluster"] == c]
    plt.scatter(d["dissolved_oxygen"], d["turbidity_ntu"], label=f"Cluster {c}", alpha=0.5)
plt.xlabel("Dissolved Oxygen")
plt.ylabel("Turbidity (NTU)")
plt.title("K-Means Clustering: Water Samples")
plt.legend()
plt.tight_layout()
plt.show()

---
## สรุปหลักสูตร AI & Data Analytics (Machine Learning)

### สิ่งที่ได้เรียนรู้ตลอดหลักสูตร

| บท | หัวข้อ | ทักษะที่ได้ |
|-----|--------|------------|
| บทที่ 1 | AI Crash Course | AI/ML/DL คืออะไร, Teachable Machine |
| บทที่ 2 | Classification | Train/Test Split, Decision Tree, KNN, Accuracy, Confusion Matrix |
| บทที่ 3 | Regression + Clustering | Linear Regression, R², K-Means |
| บทที่ 4 | Workshop | สร้าง AI Pipeline ครบวงจร |
| บทที่ 5 | เฉลย + สรุป | ทบทวนและต่อยอด |

### ML Pipeline มาตรฐาน

```
1. ตั้งโจทย์ → 2. เตรียมข้อมูล → 3. เลือก Model → 4. Train → 5. ทดสอบ → 6. ปรับปรุง
```

### อาชีพสาย AI

| อาชีพ | คำอธิบาย |
|-------|----------|
| 📊 Data Scientist | วิเคราะห์ข้อมูล สร้าง Model |
| 🤖 ML Engineer | พัฒนาระบบ AI ขนาดใหญ่ |
| 🧠 AI Researcher | วิจัยเทคนิค AI ใหม่ ๆ |
| 💼 Data Analyst | วิเคราะห์ข้อมูลธุรกิจ |
| 🏥 AI + สาขาอื่น | AI in Healthcare, Agriculture, Finance... |

### แหล่งเรียนรู้เพิ่มเติม

- **Kaggle.com** — แข่งขัน Data Science + Dataset ฟรี
- **fast.ai** — คอร์ส Deep Learning ฟรี
- **Google ML Crash Course** — คอร์ส ML ฟรีจาก Google
- **Coursera / edX** — คอร์สจากมหาวิทยาลัยชั้นนำ

---

**🎓 ขอบคุณที่ร่วมเรียนรู้ในโครงการ Coding Thailand 2026!**